# Plotting notebook

Generates the visualization PNGs for the processed candidate data.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'data').exists() and (candidate / 'src').exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root')


ROOT_DIR = find_repo_root(Path.cwd())
PROCESSED_DIR = ROOT_DIR / 'data' / 'processed'
PLOTS_DIR = PROCESSED_DIR / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(PROCESSED_DIR / 'features_final.csv')
feature_importance = pd.read_csv(PROCESSED_DIR / 'feature_importance.csv') if (PROCESSED_DIR / 'feature_importance.csv').exists() else None
predictions = pd.read_csv(PROCESSED_DIR / 'model_predictions.csv') if (PROCESSED_DIR / 'model_predictions.csv').exists() else None

plt.style.use('dark_background')
sns.set_theme(style='darkgrid')


def save_fig(name: str) -> None:
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / name, dpi=160, bbox_inches='tight')
    plt.close()


numeric_df = df.select_dtypes(include='number')

# 1. Master score distribution
plt.figure(figsize=(8, 5))
sns.histplot(df['master_score'], bins=12, kde=True, color='#6C63FF')
plt.title('Master score distribution')
plt.xlabel('master_score')
plt.ylabel('Count')
save_fig('01_master_score_distribution.png')

# 2. Correlation heatmap
plt.figure(figsize=(14, 10))
corr = numeric_df.corr(numeric_only=True)
sns.heatmap(corr, cmap='coolwarm', center=0, linewidths=0.2)
plt.title('Numeric feature correlation heatmap')
save_fig('02_correlation_heatmap.png')

# 3. Feature importance
if feature_importance is not None and not feature_importance.empty:
    top_features = feature_importance.head(10).sort_values('importance')
    plt.figure(figsize=(9, 6))
    plt.barh(top_features['feature'], top_features['importance'], color='#00D4AA')
    plt.title('Top 10 feature importances')
    plt.xlabel('Importance')
    save_fig('03_feature_importance_top10.png')

# 4-9. Composite score distributions
for idx, column in enumerate([
    'activity_score',
    'profile_quality_score',
    'career_progression_score',
    'skill_depth_score',
    'engagement_score',
    'availability_score',
], start=4):
    plt.figure(figsize=(8, 5))
    sns.histplot(df[column], bins=12, kde=True, color='#FFD166')
    plt.title(f'{column} distribution')
    plt.xlabel(column)
    plt.ylabel('Count')
    save_fig(f'{idx:02d}_{column}_distribution.png')

# 10. Predicted vs actual master score
if predictions is not None and {'actual_master_score', 'predicted_master_score'}.issubset(predictions.columns):
    plot_df = predictions.dropna(subset=['predicted_master_score']).copy()
    plt.figure(figsize=(7, 7))
    sns.scatterplot(
        data=plot_df,
        x='actual_master_score',
        y='predicted_master_score',
        hue='split',
        palette='Set2',
        s=60,
        alpha=0.85,
    )
    max_score = max(plot_df['actual_master_score'].max(), plot_df['predicted_master_score'].max())
    min_score = min(plot_df['actual_master_score'].min(), plot_df['predicted_master_score'].min())
    plt.plot([min_score, max_score], [min_score, max_score], linestyle='--', color='white', linewidth=1)
    plt.title('Predicted vs actual master score')
    plt.xlabel('Actual')
    plt.ylabel('Predicted')
    save_fig('10_predicted_vs_actual_master_score.png')

print(f'Saved plots to {PLOTS_DIR}')

Saved plots to C:\Users\Disha Satpute\Downloads\redrob-ai\data\processed\plots
